# 27. OpenAI 임베딩 전환 확인

2026-10-05 전체 2,213개를 `text-embedding-3-small` 1536차원으로 변환하고 별도 개인 DB 테이블에 저장했습니다. 기존 청킹·원문·그림·로컬 결과는 보존했습니다.

이 노트북은 `.py` 도구의 역할을 작은 단계로 확인합니다. 첫 두 코드 셀은 로컬 결과를 읽기만 합니다. API 호출·DB 저장은 아래에 구분합니다.


In [ ]:
# [프로젝트 적용] 현재 작업 폴더가 어디든 프로젝트 src를 찾습니다.
from pathlib import Path
import sys
import json

현재폴더 = Path.cwd()
프로젝트폴더 = next(폴더 for 폴더 in [현재폴더, *현재폴더.parents]
                    if (폴더 / 'src/car_search_rag/zzong_santafe_lag').is_dir())
if str(프로젝트폴더 / 'src') not in sys.path:
    sys.path.insert(0, str(프로젝트폴더 / 'src'))

from car_search_rag.zzong_santafe_lag.openai_migration import load_plan, load_batches
계획, 결과폴더 = load_plan()
print('모델:', 계획['manifest']['model_name'])
print('벡터 차원:', 계획['manifest']['dimension'])
print('기존 청크 수:', len(계획['manifest']['rows']))
print('문서 예상 비용(달러):', 계획['estimated_document_cost_usd'])
print('아직 API 호출·DB 조회 없음')


## 1. 이미 만든 벡터 확인

벡터는 검색에 쓰는 숫자 목록입니다. 아래 셀은 저장한 중간 결과를 읽어 개수·차원·정규화만 검사합니다. 전체 숫자를 화면에 출력하지 않습니다.


In [ ]:
벡터 = load_batches(계획, 결과폴더)
print('벡터 배열 크기:', 벡터.shape)
저장결과 = json.loads((결과폴더 / 'db_save_result.json').read_text(encoding='utf-8'))
print('새 임베딩 작업 번호:', 저장결과['embedding_run_id'])
print('DB 대조 완료 벡터 수:', 저장결과['vector_count'])
print('기존 원문·로컬 벡터·그림 상태 유지:', 저장결과['source_unchanged'])


## 2. Python 전환 도구의 실행 순서

`openai_migration.py`의 `prepare()` → `embed(sample=True)` → `embed()` → `save()` 순서입니다. 이번 작업은 이미 완료됐습니다. 다시 임베딩하지 않아도 됩니다.

- prepare: 기존 DB를 읽어 입력 글과 비용 확인
- sample: 16개를 API로 먼저 확인
- embed: 나머지를 API로 계산하며 성공 구간 저장
- save: 새 테이블에만 저장하고 다시 읽어 확인

기존 입력 글이 바뀌면 이전 체크포인트를 재사용하지 않습니다. 자동으로 글을 자르거나 검토 상태를 바꾸지 않습니다.


## 3. 새 검색으로 질문 한 개 확인 — 질문 임베딩 API 호출

아래 셀을 실행하면 **질문을 OpenAI로 보내 임베딩**합니다. 답변 생성 API는 호출하지 않습니다. 원문·그림 연결은 기존 개인 자료를 읽습니다.


In [ ]:
# [모델 추가] 문서와 질문을 같은 OpenAI 모델·1536차원으로 비교합니다.
from car_search_rag.zzong_santafe_lag.openai_search_service import OpenAIManualSearchService, active_embedding_run
from car_search_rag.zzong_santafe_lag.answer_service import ManualAnswerService

검색서비스 = OpenAIManualSearchService(active_embedding_run())
발췌서비스 = ManualAnswerService(저장결과['source_run_id'], search_service=검색서비스)
발췌 = 발췌서비스.answer('차대번호가 새겨진 위치를 그림으로 보여줘.', top_k=5, progress=print)
print('상태:', 발췌['status'])
print('질문 임베딩 API:', 발췌['query_embedding_api_called'])
print('답변 생성 API:', 발췌['llm_called'])
print('연결된 표시 가능 그림 수:', len(발췌['images']))
print(발췌['answer'])


## 4. 기존 30문항 비교 결과 읽기 — API 호출 없음

Hit@5·MRR@5는 같은 정답 목록의 출처 순위를 비교합니다. `Precision@5_expected_ids`는 정답 ID 목록 기준의 보조값이며, 전체 관련 근거를 판정한 정식 Precision@5는 아직 아닙니다.


In [ ]:
비교파일 = 결과폴더 / 'comparison_result.json'
if 비교파일.exists():
    비교 = json.loads(비교파일.read_text(encoding='utf-8'))
    print('질문 수:', 비교['question_count'])
    print(json.dumps(비교['metrics'], ensure_ascii=False, indent=2))
    for 한계 in 비교['limitations']:
        print('-', 한계)
else:
    print('전체 비교가 완료되지 않았습니다. 중간 결과를 완료 점수로 표시하지 않습니다.')


## 근거·보류 확인 결과
이전 미업로드 조건 기준은 9/10입니다. A10의 그림은 이미 업로드돼 연결됩니다. 업로드 완료 조건으로 A10만 갱신한 v2는 같은 저장 답변 재채점 10/10이며 두 결과를 보존했습니다. 답변 생성 API는 호출하지 않았습니다.


## 다음 확인

기존 `run_app.bat`로 화면 서버를 다시 실행하고 왼쪽 **검색에 사용할 모델 → OpenAI**를 확인하세요. 답변 생성 모델은 기존 `gpt-6-luna`이며, 생성 버튼을 누를 때만 호출합니다.

설계·재개 명령·비용·DBeaver 구조 설명은 같은 폴더의 `openai_embedding_guide.md`에 있습니다.
